# 02 — mZEB Concentration Evolution

Attractor perturbation sweep — 105 trajectories (35 per attractor).

> Run notebook 01 first to generate `results/data/bifurcation_results.npz`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.integrate import odeint
from google.colab import output
output.enable_custom_widget_manager()

from src.ode.emt_ode import ode_system
from src.utils.plotting import set_dark_theme

set_dark_theme()
print("Setup complete.")


## Load bifurcation results

In [ ]:
d = np.load('../results/data/bifurcation_results.npz')
res_fwd, res_bwd, res_mid = d['res_fwd'], d['res_bwd'], d['res_mid']
I_fwd,   I_bwd,   I_mid   = d['I_fwd'],  d['I_bwd'],  d['I_mid']
print(f"Loaded: res_fwd {res_fwd.shape}, res_bwd {res_bwd.shape}, res_mid {res_mid.shape}")


## Interactive attractor perturbation widget

35 perturbation scales (0.1× – 3.0×) per attractor → 105 trajectories total.
- Left: mZEB trajectories coloured by originating attractor
- Right: final steady-state mZEB vs perturbation scale

In [ ]:
import os
os.makedirs('../results/figures', exist_ok=True)

t_span = np.linspace(0, 500, 2000)

def run_and_plot(I_fixed_k):
    I_fixed = I_fixed_k * 1e3

    idx_fwd = np.argmin(np.abs(I_fwd - I_fixed))
    idx_bwd = np.argmin(np.abs(I_bwd - I_fixed))
    idx_mid = np.argmin(np.abs(I_mid - I_fixed))

    x0_E      = res_fwd[idx_fwd].copy(); x0_E[6]      = I_fixed
    x0_M      = res_bwd[idx_bwd].copy(); x0_M[6]      = I_fixed
    x0_hybrid = res_mid[idx_mid].copy(); x0_hybrid[6] = I_fixed

    perturbations = np.linspace(0.1, 3.0, 35)
    attractor_info = [
        ('E',      x0_E,      'royalblue'),
        ('M',      x0_M,      'tomato'),
        ('Hybrid', x0_hybrid, 'seagreen'),
    ]

    trajectories = []
    labels       = []
    total = len(attractor_info) * len(perturbations)
    count = 0

    print(f"Integrating {total} trajectories at I = {I_fixed/1e3:.0f}×10³...")

    for label, x0_base, _ in attractor_info:
        for scale in perturbations:
            x0      = x0_base.copy()
            x0[:6] *= scale
            x0[6]   = I_fixed
            x0      = np.clip(x0, 0, None)
            sol = odeint(ode_system, x0, t_span, rtol=1e-8, atol=1e-10)
            trajectories.append(sol[:, 1])
            labels.append(label)
            count += 1
            if count % 10 == 0 or count == total:
                print(f"  [{count}/{total}] done")

    print("All integrations complete.")

    labels     = np.array(labels)
    final_vals = np.array([traj[-1] for traj in trajectories])

    for label, _, color in attractor_info:
        mask   = labels == label
        median = np.median(final_vals[mask])
        spread = final_vals[mask].max() - final_vals[mask].min()
        print(f"  {label:6s} — median mZEB = {median:.2f},  spread = {spread:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    ax1 = axes[0]
    for i, (traj, label) in enumerate(zip(trajectories, labels)):
        color = [c for l, _, c in attractor_info if l == label][0]
        ax1.plot(t_span, traj, color=color, lw=0.9, alpha=0.4,
                 label=label if i in [0, len(perturbations), 2*len(perturbations)]
                 else '_nolegend_')

    for label, x0_base, color in attractor_info:
        mask   = labels == label
        ss_val = np.median(final_vals[mask])
        ax1.axhline(ss_val, ls='--', lw=1.2, color=color, alpha=0.8,
                    label=f'{label}: {ss_val:.1f}')

    ax1.set_xlabel('Time (a.u.)', fontsize=12)
    ax1.set_ylabel('mZEB concentration (a.u.)', fontsize=12)
    ax1.set_title(f'mZEB Evolution — Perturbed Attractors  (I = {I_fixed/1e3:.0f}×10³)', fontsize=13)
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    ax2 = axes[1]
    for label, _, color in attractor_info:
        mask = labels == label
        ax2.scatter(perturbations, final_vals[mask], color=color, s=40, label=label, zorder=3)
        ss_val = np.median(final_vals[mask])
        ax2.axhline(ss_val, ls='--', lw=1.2, color=color, alpha=0.7)

    ax2.set_xlabel('Perturbation scale (× attractor)', fontsize=12)
    ax2.set_ylabel('Final mZEB steady state (a.u.)', fontsize=12)
    ax2.set_title('Basin Robustness — Do Trajectories Stay in Their Basin?', fontsize=13)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'../results/figures/mZEB_tristability_I{int(I_fixed/1e3)}k.png',
                dpi=200, bbox_inches='tight')
    plt.show()


slider = widgets.FloatSlider(
    value=55, min=20, max=120, step=5,
    description='I (×10³):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
widgets.interact_manual(run_and_plot, I_fixed_k=slider)
